# Research Question 1: 

The goal: for each ecoregion, estimate how strongly and in which direction pre-season climate anomaly predicts fire season onset timing (β, days/°C), using statsmodels.MixedLM with partial pooling across ecoregions.


In [ ]:
# Imports

import pandas as pd
import numpy as np
from statsmodels.regression.mixed_linear_model import MixedLM
import os
import warnings
import logging
from statsmodels.tools.sm_exceptions import ConvergenceWarning
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import json
from shapely.geometry import shape
import geopandas as gpd

print('Libraries loaded.')

pd.set_option('display.expand_frame_repr', False)

## Versioning

In [ ]:
# !!!IMPORTANT!!! 
# Folders and some file names are dependent on this

RUN_LABEL   = 'med_basin'  # short name for this run
RUN_VERSION = 'v4'         # increment this for each new run
RUN_NAME   = f'{RUN_LABEL}_{RUN_VERSION}'

In [ ]:
# Config (paths, grid definitions)

RESPONSE_VARS  = ['onset_doy', 'peak_doy', 'end_doy', 'season_length']
LAG_WINDOWS    = [30, 45, 60, 90]
PREDICTOR_SETS = {
    'temp':      ['temp_anomaly_{lag}d'],
    'precip':    ['precip_anomaly_{lag}d'],
    'both':      ['temp_anomaly_{lag}d', 'precip_anomaly_{lag}d'],
}
GROUP_COL = 'eco_id'

# File paths

# BASE_OUT_DIR = 'C:/Users/ibekar/Documents/GitProjects/TGPF' # Windows
BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac

input_dir_processed = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'inputs')
results_dir = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'outputs', 'RQ1')
plot_dir = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'outputs', 'RQ1', 'plots')
os.makedirs(results_dir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)

print(f'Base output directory: {BASE_OUT_DIR}')
print(f'Input directory: {input_dir_processed}')
print(f'Results directory: {results_dir}')
print(f'Plot directory: {plot_dir}')



In [ ]:
# Geometries

geo_path = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'eco_geometries.json')

with open(geo_path) as f:
    data = json.load(f)

gdf = gpd.GeoDataFrame(
    [{'eco_id': d['eco_id'], 'eco_name': d['eco_name']} for d in data],
    geometry=[shape(d['geometry']) for d in data],
    crs='EPSG:4326'
)
print(f'Loaded {len(gdf)} ecoregion geometries.')

In [ ]:
# Load files
df = pd.read_csv(os.path.join(input_dir_processed, 'era5_anomalies_by_lag.csv'))

print(df)
print()
print("NA Check")
print(df.isnull().sum())

In [ ]:
df[df.isnull().any(axis=1)]

df = df.dropna(subset=[col for col in df.columns if 'anomaly' in col])
print(df.isnull().sum())

In [ ]:
# Model fitting function

def fit_mixedlm(df, response, predictors, group_col):
    df_subset = df[[response] + predictors + [group_col]].dropna()

    # Standardize predictors
    scaler_parameters = {}
    for predictor in predictors:
        mean = df_subset[predictor].mean()
        std = df_subset[predictor].std()
        df_subset[predictor + '_z'] = (df_subset[predictor] - mean) / std
        scaler_parameters[predictor] = {
            'mean': mean,
            'std': std
        }

    z_predictors = [pred + '_z' for pred in predictors]
    formula = response + ' ~ ' + ' + '.join(z_predictors)
    re_formula = ' ~ ' + ' + '.join(z_predictors)
    fallback = False

    try:
        model = MixedLM.from_formula(formula, groups=df_subset[group_col], re_formula=re_formula, data=df_subset)
        results = model.fit(reml=False)
        convergence_ok = results.converged
    except Exception as e:
        print(f"Model fitting failed for {response} with predictors {predictors}. Attempting fallback to random intercept only.")
        fallback = True
        model = MixedLM.from_formula(formula, groups=df_subset[group_col], data=df_subset)
        results = model.fit()
        convergence_ok = results.converged

    # Extract fixed effects
    ci = results.conf_int()
    fixed_effects = {}
    for zp in z_predictors:
        original_name = zp[:-2]  # Remove '_z' suffix
        coef = results.params[zp]
        se = results.bse[zp]
        pval = results.pvalues[zp]
        ci_low = ci.loc[zp, 0]
        ci_hi = ci.loc[zp, 1]

        fixed_effects[original_name] = {
            'coef' : coef,
            'se'   : se,
            'pval' : pval,
            'ci_low': ci_low,
            'ci_hi' : ci_hi,
        }

    # get eco_id and re_dict for random effects
    random_effects = []
    for group, re_params in results.random_effects.items():
        row = {'eco_id': group}
        for zp in z_predictors:
            total_slope = fixed_effects[zp[:-2]]['coef'] + re_params.get(zp, 0)
            row[zp[:-2] + '_slope'] = total_slope
        random_effects.append(row)

    # Extract model fit metrics
    aic = results.aic
    log_likelihood = results.llf

    # Back-transform slopes to original units
    for fe in fixed_effects:
        std = scaler_parameters[fe]['std']
        fixed_effects[fe]['coef_unscaled'] = fixed_effects[fe]['coef'] / std
        fixed_effects[fe]['ci_low_unscaled'] = fixed_effects[fe]['ci_low'] / std
        fixed_effects[fe]['ci_hi_unscaled'] = fixed_effects[fe]['ci_hi'] / std

    for row in random_effects:
        for pred in predictors:
            std = scaler_parameters[pred]['std']
            row[pred + '_slope_original'] = row[pred + '_slope'] / std

    return {
        'fixed_effects':  fixed_effects,
        'random_effects': random_effects,
        'aic':            aic,
        'log_likelihood': log_likelihood,
        'fallback':       fallback,
        'response':       response,
        'predictors':     predictors,
        'convergence_ok': convergence_ok,
        }


In [ ]:
# Warning logs

# Remove existing handlers
root_logger = logging.getLogger()
for handler in root_logger.handlers[:]:
    root_logger.removeHandler(handler)
    handler.close()

# Now basicConfig will take effect
logging.basicConfig(
    filename = os.path.join(results_dir, 'rq1_convergence_log.txt'), 
    level= logging.WARNING,
    format= '%(asctime)s - %(message)s',
    filemode= 'w'
)

warnings.filterwarnings('always', category=ConvergenceWarning)

current_model_context = {}

def custom_warning(msg, *args, **kwargs):
    context = current_model_context
    log_msg = (
        f"response={context.get('response','?')} | "
        f"lag={context.get('lag','?')} | "
        f"pred_set={context.get('pred_set','?')} | "
        f"{str(msg)}"
    )
    logging.warning(log_msg)
    # Short readable print to notebook
    print(f"  ⚠ Convergence issue: {context.get('response','?')} | lag={context.get('lag','?')} | {context.get('pred_set','?')}")

warnings.showwarning = custom_warning

In [ ]:
# Modeling loop

all_fixed = []
all_random = []

for response in RESPONSE_VARS:
    for lag in LAG_WINDOWS:
        for pred_set_name, pred_template_list in PREDICTOR_SETS.items():

            # Resolve actual column names
            predictors = [p.format(lag=lag) for p in pred_template_list]

            # Progress message
            print(f"Running: response={response} | lag={lag} | pred_set={pred_set_name}")

            #Update the model context for logging
            current_model_context = {'response': response, 'lag': lag, 'pred_set': pred_set_name}

            # Fit model
            run_result = fit_mixedlm(df, response, predictors, GROUP_COL)

            # Build fixed effects summary row
            fixed_row = {
                'response':       response,
                'lag':            lag,
                'pred_set':       pred_set_name,
                'fallback':       run_result['fallback'],
                'aic':            run_result['aic'],
                'log_likelihood': run_result['log_likelihood'],
            }
            for pred, vals in run_result['fixed_effects'].items():
                fixed_row[pred + '_coef']          = vals['coef']
                fixed_row[pred + '_coef_unscaled'] = vals['coef_unscaled']
                fixed_row[pred + '_se']            = vals['se']
                fixed_row[pred + '_pval']          = vals['pval']
                fixed_row[pred + '_ci_low']        = vals['ci_low']
                fixed_row[pred + '_ci_hi']         = vals['ci_hi']
                fixed_row[pred + '_ci_low_unscaled'] = vals['ci_low_unscaled']
                fixed_row[pred + '_ci_hi_unscaled']  = vals['ci_hi_unscaled']
                fixed_row['convergence_ok'] = run_result['convergence_ok']
            all_fixed.append(fixed_row)

            # Build random effects rows
            for row in run_result['random_effects']:
                row['response'] = response
                row['lag']      = lag
                row['pred_set'] = pred_set_name
                all_random.append(row)

print(f"\nDone. {len(all_fixed)} model runs completed.")

In [ ]:
# Tidy format and write outputs 
df_fixed  = pd.DataFrame(all_fixed)
df_random = pd.DataFrame(all_random)

# Random df
id_cols = ["eco_id", "response", "lag", "pred_set"]
value_cols = [col for col in df_random.columns if col.endswith("_slope_original")]

df_random_tidy = df_random.melt(id_vars = id_cols,
                               value_vars = value_cols,
                               var_name = "predictors",
                               value_name = "slope").dropna(subset="slope")

df_random_tidy["predictors"] = df_random_tidy["predictors"].str.replace("_slope_original", "")

print(df_random_tidy)

# Fixed df

# Tidy fixed effects
fixed_id_cols = ['response', 'lag', 'pred_set', 'fallback', 'convergence_ok', 'aic', 'log_likelihood']

metrics = ['coef', 'coef_unscaled', 'se', 'pval', 'ci_low', 'ci_hi', 'ci_low_unscaled', 'ci_hi_unscaled']

tidy_parts = []
for metric in metrics:
    value_cols_fixed = [col for col in df_fixed.columns if col.endswith(f'_{metric}')]
    melted = df_fixed.melt(
        id_vars=fixed_id_cols,
        value_vars=value_cols_fixed,
        var_name='predictor',
        value_name=metric
    ).dropna(subset=metric)
    melted['predictor'] = melted['predictor'].str.replace(f'_{metric}', '', regex=False)
    tidy_parts.append(melted)

# Merge all metrics together on common columns
merge_cols = fixed_id_cols + ['predictor']
df_fixed_tidy = tidy_parts[0]
for part in tidy_parts[1:]:
    df_fixed_tidy = df_fixed_tidy.merge(part, on=merge_cols, how='outer')

print(df_fixed_tidy)

In [ ]:
df_fixed_tidy.to_csv(os.path.join(results_dir, 'rq1_fixed_effects_tidy.csv'), index=False)
df_random_tidy.to_csv(os.path.join(results_dir, 'rq1_random_effects_tidy.csv'), index=False)


print(f'Fixed effects table:  {df_fixed.shape}')
print(f'Random effects table: {df_random.shape}')

In [ ]:
df_converged = df_fixed_tidy[df_fixed_tidy['convergence_ok'] == True]
print(f'{len(df_converged)} converged runs out of 48')

In [ ]:
temp_runs = df_fixed_tidy[
    (df_fixed_tidy['pred_set'] == 'temp') &
    (df_fixed_tidy['convergence_ok'] == True)
][['response', 'lag', 'predictor', 'coef_unscaled', 'pval', 'convergence_ok']]

print(temp_runs.to_string())

In [ ]:
precip_runs = df_fixed_tidy[
    (df_fixed_tidy['pred_set'] == 'precip') &
    (df_fixed_tidy['convergence_ok'] == True)
][['response', 'lag', 'predictor', 'coef_unscaled', 'pval', 'convergence_ok']]

print(precip_runs.to_string())

# Visualisation

In [ ]:
heat_df = df_fixed_tidy[
    (df_fixed_tidy['pred_set'] == 'temp') &
    (df_fixed_tidy['convergence_ok'] == True)
].copy()

pivot = heat_df.pivot(index='response', columns='lag', values='coef_unscaled')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdBu', center=0, ax=ax)
ax.set_title('Fixed effect temperature slope (days/°C) by response and lag window')
ax.set_xlabel('Lag window (days)')
ax.set_ylabel('Response variable')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'RQ1_03_heatmap_fixed_effects.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def plot_histogram(df_random_tidy, response, lag, pred_set, results_dir, all_lags):
    
    # Compute global x range and max count across all lags
    all_slopes = []
    all_counts = []
    for l in all_lags:
        subset = df_random_tidy[
            (df_random_tidy['response'] == response) &
            (df_random_tidy['lag'] == l) &
            (df_random_tidy['pred_set'] == pred_set) &
            (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{l}d')
        ]
        all_slopes.append(subset['slope'])
        counts, _ = np.histogram(subset['slope'], bins=20)
        all_counts.append(counts.max())
    
    all_slopes = pd.concat(all_slopes)
    x_min = all_slopes.min()
    x_max = all_slopes.max()
    global_max = max(all_counts)

    # Filter to current lag
    plot_df = df_random_tidy[
        (df_random_tidy['response'] == response) &
        (df_random_tidy['lag'] == lag) &
        (df_random_tidy['pred_set'] == pred_set) &
        (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{lag}d')
    ]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(plot_df['slope'], bins=20, edgecolor='black')
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='zero')
    ax.set_xlim(x_min - 0.5, x_max + 0.5)
    ax.set_ylim(0, global_max + 1)
    ax.set_xlabel('Per-ecoregion slope (days/°C)')
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {pred_set} sensitivity of {response} ({lag}d lag)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'RQ1_01_histogram_{response}_{pred_set}_{lag}d.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for lag in LAG_WINDOWS:
    plot_histogram(df_random_tidy, 'onset_doy', lag, 'temp', plot_dir, LAG_WINDOWS)

In [ ]:
def plot_forest(df_random_tidy, df, response, lag, pred_set, results_dir):
    
    plot_df = df_random_tidy[(df_random_tidy["response"] == response) &
                             (df_random_tidy["lag"] == lag) &
                             (df_random_tidy["pred_set"] == pred_set) &
                             (df_random_tidy["predictors"] == f'{pred_set}_anomaly_{lag}d')]
    
    # Forest plot temp sensitivity
    eco_meta = df[['eco_id', 'eco_name', 'biome_name']].drop_duplicates()
    forest_df = plot_df[['eco_id', 'slope']].merge(eco_meta, on='eco_id')
    forest_df = forest_df.sort_values('slope')

    biomes = forest_df['biome_name'].unique()
    palette = plt.cm.tab10.colors
    biome_color_map = {biome: palette[i % len(palette)] for i, biome in enumerate(biomes)}
    colors = [biome_color_map[b] for b in forest_df['biome_name']]

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(range(len(forest_df)), forest_df['slope'], color=colors)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_yticks(range(len(forest_df)))
    ax.set_yticklabels(forest_df['eco_name'], fontsize=7)
    ax.set_xlabel('Per-ecoregion slope (days/°C)')
    ax.set_title(f'Temperature sensitivity of fire onset per ecoregion ({lag}d lag)')

    # Legend
    legend_elements = [Patch(facecolor=biome_color_map[b], label=b) for b in biomes]
    ax.legend(handles=legend_elements, fontsize=7, loc='upper left')

    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f'RQ1_02_forestplot_{response}_{pred_set}_{lag}d.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for lag in LAG_WINDOWS:
    plot_forest(df_random_tidy, df, 'onset_doy', lag, 'temp', plot_dir)

In [ ]:
def plot_map(df_random_tidy, df, gdf, response, lag, pred_set, results_dir, all_lags):
    
    # Compute global slope range across all lags for consistent colormap
    all_slopes = []
    for l in all_lags:
        subset = df_random_tidy[
            (df_random_tidy['response'] == response) &
            (df_random_tidy['lag'] == l) &
            (df_random_tidy['pred_set'] == pred_set) &
            (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{l}d')
        ]
        all_slopes.append(subset['slope'])
    
    all_slopes = pd.concat(all_slopes)
    vmin = all_slopes.min()
    vmax = all_slopes.max()

    # Filter to current lag
    plot_df = df_random_tidy[
        (df_random_tidy['response'] == response) &
        (df_random_tidy['lag'] == lag) &
        (df_random_tidy['pred_set'] == pred_set) &
        (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{lag}d')
    ]

    # Merge with metadata and geometries
    eco_meta = df[['eco_id', 'biome_name']].drop_duplicates()
    plot_df = plot_df.merge(eco_meta, on='eco_id')
    map_df = gdf.merge(plot_df[['eco_id', 'slope', 'biome_name']], on='eco_id')

    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))
    map_df.plot(
        column='slope',
        cmap='RdBu',
        legend=True,
        vmin=vmin,
        vmax=vmax,
        legend_kwds={'label': 'Per-ecoregion slope (days/°C)', 'shrink': 0.6},
        edgecolor='black',
        linewidth=0.3,
        ax=ax
    )
    ax.set_title(f'{pred_set} sensitivity of {response} per ecoregion ({lag}d lag)', fontsize=13)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f'RQ1_04_map_{response}_{pred_set}_{lag}d.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for lag in LAG_WINDOWS:
    plot_map(df_random_tidy, df, gdf, 'onset_doy', lag, 'temp', plot_dir, LAG_WINDOWS)

In [ ]:
def plot_lag_profile(df_fixed_tidy, pred_set, results_dir):
    
    plot_df = df_fixed_tidy[
        (df_fixed_tidy['pred_set'] == pred_set) &
        (df_fixed_tidy['convergence_ok'] == True)
    ]

    fig, ax = plt.subplots(figsize=(8, 5))
    
    for response in plot_df['response'].unique():
        response_df = plot_df[plot_df['response'] == response].sort_values('lag')
        ax.plot(response_df['lag'], response_df['coef_unscaled'], 
                marker='o', label=response)
        ax.fill_between(response_df['lag'],
                        response_df['ci_low_unscaled'],
                        response_df['ci_hi_unscaled'],
                        alpha=0.15)

    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.set_xlabel('Lag window (days)')
    ax.set_ylabel('Fixed effect slope (days/°C)')
    ax.set_title(f'Climate sensitivity across lag windows — {pred_set}')
    ax.set_xticks([30, 45, 60, 90])
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f'RQ1_05_lag_profile_{pred_set}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    

In [ ]:
plot_lag_profile(df_fixed_tidy, 'temp', plot_dir)
plot_lag_profile(df_fixed_tidy, 'precip', plot_dir)

In [ ]:
def plot_aic(df_fixed_tidy, response, plot_dir):
    
    plot_df = df_fixed_tidy[
        (df_fixed_tidy['response'] == response) &
        (df_fixed_tidy['convergence_ok'] == True)
    ].drop_duplicates(subset=['response', 'lag', 'pred_set'])

    fig, ax = plt.subplots(figsize=(9, 5))

    pred_sets = plot_df['pred_set'].unique()
    lags = sorted(plot_df['lag'].unique())
    n_preds = len(pred_sets)
    bar_width = 0.25
    x = range(len(lags))

    for i, pred_set in enumerate(pred_sets):
        pred_df = (
            plot_df[plot_df['pred_set'] == pred_set]
            .set_index('lag')
            .reindex(lags)  # align to full lag list, fills missing with NaN
        )
        offsets = [xi + i * bar_width for xi in x]
        ax.bar(offsets, pred_df['aic'], width=bar_width, label=pred_set)

    ax.set_xticks([xi + bar_width for xi in x])
    ax.set_xticklabels([f'{l}d' for l in lags])
    ax.set_xlabel('Lag window (days)')
    ax.set_ylabel('AIC')
    ax.set_title(f'AIC comparison by lag and predictor set — {response}')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f'RQ1_06_aic_{response}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for response in RESPONSE_VARS:
    plot_aic(df_fixed_tidy, response, plot_dir)